# Alkahest 0.8B Heretic RP SFT

This notebook fine-tunes `thomasjvu/alkahest-0.8b-heretic-merged` on the generated `roleplay_v2` corpus, saves a LoRA adapter, attempts a merged Hugging Face checkpoint, and leaves outputs under `/kaggle/working/alkahest-08b-sft`.

In [ ]:
import os, platform, shutil, subprocess, textwrap
from pathlib import Path

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major < 7:
            raise RuntimeError('Kaggle assigned a pre-Volta GPU. This notebook needs T4 or newer; push with --accelerator NvidiaTeslaT4.')
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    if 'pre-Volta' in str(exc):
        raise

In [ ]:
import os, subprocess, sys

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

packages = [
    'git+https://github.com/huggingface/transformers.git',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/alkahest-ai/heretic-to-onnx.git'
REPO_REF = 'codex/kaggle-heretic-2b-run'
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')

if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])

print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-sft')
DATASET_JSONL = WORK_DIR / 'master-5000-generated.jsonl'
SPLIT_DIR = WORK_DIR / 'splits'
ADAPTER_DIR = WORK_DIR / 'adapter'
MERGED_DIR = WORK_DIR / 'alkahest-0.8b-heretic-rp-sft-merged'
WORK_DIR.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts/review_table_to_jsonl.py'),
    '--input', str(REPO_DIR / 'data/roleplay_v2/review_table/master-5000.tsv'),
    '--source-jsonl', str(REPO_DIR / 'data/roleplay_v2/generated_raw/master-5000.jsonl'),
    '--output', str(DATASET_JSONL),
    '--include-non-approved',
])
subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts/prepare_roleplay_dataset.py'),
    '--input', str(DATASET_JSONL),
    '--output-dir', str(SPLIT_DIR),
    '--val-fraction', '0.02',
])

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
WORK_DIR = Path('/kaggle/working/alkahest-08b-sft')
SPLIT_DIR = WORK_DIR / 'splits'
ADAPTER_DIR = WORK_DIR / 'adapter'
MERGED_DIR = WORK_DIR / 'alkahest-0.8b-heretic-rp-sft-merged'

cmd = [
    sys.executable,
    str(REPO_DIR / 'scripts/train_alkahest_sft.py'),
    '--model-name', 'thomasjvu/alkahest-0.8b-heretic-merged',
    '--train-file', str(SPLIT_DIR / 'train.jsonl'),
    '--val-file', str(SPLIT_DIR / 'val.jsonl'),
    '--dataset-manifest', str(SPLIT_DIR / 'manifest.json'),
    '--output-dir', str(ADAPTER_DIR),
    '--merged-output-dir', str(MERGED_DIR),
    '--max-seq-length', '1024',
    '--max-steps', os.environ.get('SFT_MAX_STEPS', '250'),
    '--gradient-accumulation-steps', os.environ.get('SFT_GRAD_ACCUM', '8'),
    '--learning-rate', os.environ.get('SFT_LR', '1e-4'),
    '--eval-steps', '50',
    '--save-steps', '50',
]
print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json

WORK_DIR = Path('/kaggle/working/alkahest-08b-sft')
for path in sorted(WORK_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(WORK_DIR), round(path.stat().st_size / 1024**2, 2), 'MB')
report = WORK_DIR / 'adapter' / 'training-run.json'
if report.exists():
    print(report.read_text()[:4000])